# IC-0921 — How diversified is your client, really?

**ML & FinTech · 115-1 · 20260921 · 50 minutes: 40 working, 10 discussion**

| Step | Min | What you do |
|---|---|---|
| 1 | 16 | Get to know the data, and learn the two clustering tools |
| 2 | 14 | Cluster the client's portfolio, two ways |
| 3 | 10 | Give the client advice |
| — | 10 | Discussion |

You work at a robo-advisor. A client calls:

> *"I'm well diversified. I own twelve different stocks."*

Twelve tickers is not twelve bets. If several of them rise and fall together, the client owns
fewer independent positions than he thinks. **By the end you will tell him how many bets he
actually owns.**

You may use any AI tool. It will write the code. The code is not what is graded — whether you
understand what the output means is.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

---
# Step 1 — Know your data, learn the tools (16 min)

## 1a. What the client owns

Run the cell. The table gives every ticker, the company behind it, and its industry.
**Keep this table in view for the rest of the exercise.**

In [ ]:
CAND = ["historical_stock_prices_2017.csv",
        "../slides/historical_stock_prices_2017.csv",
        "../00-course-info/in-class-exercise/historical_stock_prices_2017.csv"]
path = next((p for p in CAND if Path(p).exists()), None)
if path is None:
    raise FileNotFoundError("Put historical_stock_prices_2017.csv next to this notebook.")

prices = pd.read_csv(path, index_col=0)
prices.index = pd.to_datetime(prices.index)

holdings = pd.DataFrame([
    ("BANC", "Banc of California",        "Banking"),
    ("BANF", "BancFirst Corporation",     "Banking"),
    ("GWB",  "Great Western Bancorp",     "Banking"),
    ("ISTR", "Investar Holding Corp.",    "Banking"),
    ("CAKE", "The Cheesecake Factory",    "Restaurants"),
    ("DFRG", "Del Frisco's Restaurant Gp.","Restaurants"),
    ("LOCO", "El Pollo Loco Holdings",    "Restaurants"),
    ("BREW", "Craft Brew Alliance",       "Brewing"),
    ("AAPL", "Apple Inc.",                "Technology"),
    ("EBAY", "eBay Inc.",                 "E-commerce"),
    ("QQQ",  "Invesco QQQ Trust",         "ETF, Nasdaq-100"),
    ("CCOI", "Cogent Communications",     "Telecom"),
], columns=["ticker", "company", "industry"]).set_index("ticker")

PORTFOLIO = holdings.index.tolist()
P = prices[PORTFOLIO]          # 251 trading days of 2017 x 12 stocks
print(P.shape)
holdings

## 1b. Prices through the year

In [ ]:
P.plot(figsize=(11, 5), linewidth=1)
plt.title("Daily closing prices, 2017")
plt.ylabel("price (USD)")
plt.legend(ncol=4, fontsize=8)
plt.show()

## 1c. Daily returns

Prices tell you the level. **Returns tell you the movement**, which is what risk is made of.

In [ ]:
R = np.log(P).diff().dropna()     # daily log returns

R.plot(subplots=True, layout=(4, 3), figsize=(12, 8),
       sharey=True, legend=False, linewidth=0.7, title=list(R.columns))
plt.tight_layout()
plt.show()

## 1d. Summary statistics

In [ ]:
summary = pd.DataFrame({
    "mean price":    P.mean(),
    "min price":     P.min(),
    "max price":     P.max(),
    "ann. return %": R.mean() * 252 * 100,
    "ann. vol %":    R.std() * np.sqrt(252) * 100,
}).round(1)
summary.join(holdings[["industry"]])

## 1e. Risk against reward

Each stock as one point: risk on the horizontal axis, reward on the vertical.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(summary["ann. vol %"], summary["ann. return %"], s=60)
for t, row in summary.iterrows():
    plt.annotate(t, (row["ann. vol %"], row["ann. return %"]),
                 xytext=(4, 4), textcoords="offset points")
plt.axhline(0, color="grey", linewidth=0.8)
plt.xlabel("annualised standard deviation (%)   — risk")
plt.ylabel("annualised return (%)   — reward")
plt.title("Risk and return, 2017")
plt.show()

### Questions on the data

**Q1.** Which stock has the **lowest** annualised volatility of all twelve? Look up what it is in
the holdings table. Why would that kind of holding be less volatile than a single company?

> **QQQ** (≈10% annualised vol) is the lowest. QQQ is an ETF holding the Nasdaq-100, i.e. about 100 companies, so the idiosyncratic risk of individual firms averages out; a single company (e.g. BREW ≈41%, BANC ≈37%) carries all of its own firm-specific risk.

**Q2.** Four of the twelve are banks. From the returns plots and the risk–return chart, would you
expect all four to move together? Write down your guess now — you will test it in Step 2.

> I would guess the four banks mostly move together (same industry, same interest-rate / credit exposure), so they'd sit close to each other in the charts — though maybe not perfectly, since they are small regional banks with different vol and return.

## 1f. The two tools, on something small

Before the portfolio, practise on eight credit card customers where you can see the answer with
your own eyes. The code is complete — run it and read the output.

In [ ]:
cust = pd.DataFrame({
    "customer": ["Amy", "Ben", "Cora", "Dan", "Eve", "Fay", "Gus", "Hal"],
    "spend":    [5, 6, 4, 5, 28, 30, 32, 29],   # monthly card spending, NT$ thousand
    "trans":    [4, 5, 3, 6, 22, 25, 20, 24],   # card transactions per month
}).set_index("customer")

cust["kmeans"] = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(cust)

plt.scatter(cust["spend"], cust["trans"], c=cust["kmeans"], s=100)
for name, row in cust.iterrows():
    plt.annotate(name, (row["spend"], row["trans"]))
plt.xlabel("monthly spending (NT$ thousand)"); plt.ylabel("transactions per month")
plt.show()
cust

Now the same eight customers as a **dendrogram**. Read it from the bottom up: the height at which two branches join tells you how different they are.

In [ ]:
Zc = linkage(cust[["spend", "trans"]].values, method="average")
dendrogram(Zc, labels=cust.index.tolist())
plt.ylabel("height at which groups join")
plt.show()

cust["tree"] = fcluster(Zc, 2, criterion="maxclust")
cust

**Q3.** In the dendrogram, the last two groups join very high up while everything before joined
low down. What does that big jump tell you about how many groups these customers really form?

> The big jump means the eight customers form essentially **two** natural groups: a low-spend/low-transaction group (Amy, Ben, Cora, Dan) and a high-spend/high-transaction group (Eve, Fay, Gus, Hal). Members inside each group are very similar (join low); the two groups are very different (join high). Cutting anywhere below the jump would split groups that are really one.

**Q4.** Change `n_clusters=2` to `3` and re-run 1f. Who moves? Looking at the plot, is that a real
group or is k-means forcing one?

> With K=3, **Gus** splits off on his own (Eve, Fay, Hal stay together; Amy–Dan stay together). That is not a real group: Gus is just a slightly larger spender inside the high group, and k-means is forced to produce 3 clusters, so it carves up a group that is really one. The dendrogram already told us K=2 is the right answer.

In [ ]:
# Q4 — try K = 3 here
cust3 = cust[["spend", "trans"]].copy()
cust3["kmeans3"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(cust3)
plt.scatter(cust3["spend"], cust3["trans"], c=cust3["kmeans3"], s=100)
for name, row in cust3.iterrows():
    plt.annotate(name, (row["spend"], row["trans"]))
plt.xlabel("monthly spending (NT$ thousand)"); plt.ylabel("transactions per month")
plt.title("K = 3")
plt.show()
cust3

# 2a
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(P.T)   # one row per stock, raw prices
labels_km = pd.Series(km.labels_, index=P.columns)
for k in range(3):
    members = labels_km[labels_km == k].index.tolist()
    print(f"cluster {k}: {members}")
    print(holdings.loc[members, "industry"].tolist(), "
")

In [ ]:
# 2a



# 2b
C = R.corr()                       # how much each pair moves together
D = 1 - C                          # turn correlation into a distance
Z = linkage(squareform(D.values, checks=False), method="average")
plt.figure(figsize=(9, 4))
dendrogram(Z, labels=D.columns.tolist())
plt.ylabel("distance (1 - correlation)")
plt.title("Hierarchical clustering on correlation")
plt.show()

groups = pd.Series(fcluster(Z, 3, criterion="maxclust"), index=D.columns)
for g in sorted(groups.unique()):
    print(f"group {g}: {groups[groups == g].index.tolist()}")

In [ ]:
# 2b



**Q7.** Write down your three groups.

> Group 1: **AAPL, EBAY, QQQ** (tech / Nasdaq); Group 2: **BANC, BANF, GWB, CAKE, DFRG, LOCO, BREW, CCOI** (banks, restaurants, brewer, telecom — a big cluster of small caps); Group 3: **ISTR** (alone).

**Q8.** Go back to your guess in Q2. Did the four banks end up together? If one of them did not,
name it — and note that this is a fact about the data, not a mistake.

> Three of the four banks (BANC, BANF, GWB) clustered together and are among the most correlated pairs in the portfolio (BANF–GWB ≈ 0.70). **ISTR did not**: it is nearly uncorrelated with everything (correlations ≈ 0), so it joins last, on its own. This is a fact about the 2017 data (a small, thinly-traded bank with its own idiosyncratic moves), not a mistake in the method or the code.

---
# Step 3 — Advise the client (10 min)

**Q9.** How many independent bets does the client actually own?

> Roughly **3 independent bets** at a 3-group cut (tech/Nasdaq, the small-cap US domestic cluster, and ISTR) — more like 3–4 if we also treat BREW as its own bet (it is the last stock to join the big cluster). Not twelve: the client is not as diversified as he thinks.

**Q10.** Look at AAPL and QQQ in your dendrogram, and at what QQQ is in the holdings table.
Should the client hold both? Why?

> **No, he should not hold both.** QQQ *is* the Nasdaq-100, of which AAPL is one of the largest components, so AAPL is already inside QQQ. They have the highest correlation in the portfolio (≈0.74) and join at the lowest height in the dendrogram (≈0.26). Holding both just doubles the exposure to the same bet (and EBAY, also Nasdaq-100, is in the same group).

**Q11.** He wants to cut from twelve holdings to four. **Name the four you would keep**, with one
sentence of reasoning from your own dendrogram. Four tickers — he is on the phone.

> Keep: **QQQ** , **ISTR** , **BREW** , **BANF**
>
> Because: one from each branch of the dendrogram — QQQ represents the whole tech/Nasdaq group (drop AAPL and EBAY, which it already contains), ISTR and BREW are the two stocks that move independently of everything else and so add the most diversification, and BANF stands for the cluster of banks/restaurants/telecom that move together (drop the redundant ones).

---
## Before the discussion

Restart & Run All, check every blank is filled, push as `IC-0921.ipynb`.

Be ready to answer out loud: **the client owns twelve stocks. Is he diversified?**